# YAM π0.5 inference

기본값은 MuJoCo다. `EXECUTION_TARGET='real'`이면 같은 policy server의 action을 실제 YAM에 적용하며, top/wrist CameraRig와 실제 측정 state를 사용한다. 실제 장치를 연결하기 전에는 `real`로 실행하지 않는다.

MuJoCo에서는 데이터셋의 소수 frame을 반복 사용해 server→YAM action 경로만 검증한다. 이미지와 MuJoCo 장면은 대응하지 않는다.

## YAM CAN interface 설정

Follower CAN interface를 I2RT가 사용하는 1 Mbit/s로 올립니다.

```bash
ip -br link
sudo ip link set can0 up type can bitrate 1000000
ip -details link show can0
```

In [ ]:
from collections.abc import Iterator, Mapping
from functools import partial
from itertools import cycle, islice
from pathlib import Path
import os
import signal
import socket
import subprocess
import sys
import time

import av
import numpy as np
from IPython.display import display
from PIL import Image

from yam_control import RunConfig, RuntimeDependencies, build_run_config, create_session
from yam_control.config import CameraConfig, CameraDeviceConfig, CameraFitMode, ExecutionTarget, QuestConfig, RobotConfig
from yam_control.factory import ActionProducerFactory, RobotFactory, create_default_robot
from yam_control.interfaces import ActionProducer, SafetyGate
from yam_control.policy.pi import WebsocketPiBackend, create_pi_action_producer
from yam_control.robot.i2rt_adapter import ImageProvider
from yam_control.safety import JointStepSafetyGate
from yam_control.session import RunSession
from yam_control.types import ActionChunk, EpisodeState, RobotAction, RobotObservation, RtcGuidance, SafetyDecision

# MuJoCo offscreen renderer를 초기화한다.
os.environ.setdefault('MUJOCO_GL', 'egl')

PROJECT_ROOT: Path = Path.cwd().resolve()
YAM_ABC_ROOT: Path = PROJECT_ROOT / 'third_party' / 'yam-abc-reproduce'
sys.path.insert(0, str(YAM_ABC_ROOT))
from yam_abc_reproduce.camera.interface import CameraFrame
from yam_control.camera import CameraRig
OPENPI_DIR: Path = PROJECT_ROOT / 'third_party' / 'yam-abc-reproduce' / 'third_party' / 'policy' / 'openpi'
DATASET_DIR: Path = PROJECT_ROOT / 'lerobot-v3' / 'nengjanggo' / 'yam_pick_up_the_white_ethernet_cable_and_plug_it_into_the_black_ethernet_port'
CHECKPOINT_DIR: Path = OPENPI_DIR / 'checkpoints' / 'pi05_yam_dit_lora' / 'white_ethernet_cable_dit_lora' / '14999'

EXECUTION_TARGET: ExecutionTarget = 'real'

USE_RTC: bool = True

# USE_RTC=False일 때 chunk(50개)마다 앞에서부터 실행할 action 수, None이면 chunk 전체를 실행한다.
# 작을수록 새 관측을 자주 반영하지만 재추론(약 110ms) 동안 로봇이 멈추는 횟수가 늘어난다.
OPEN_LOOP_EXECUTE_STEPS: int | None = 10

# USE_RTC=True일 때 다음 chunk를 요청하는 간격(step), 이전 chunk는 index RTC_EXECUTE_HORIZON부터 끝까지 guidance로 사용한다.
RTC_EXECUTE_HORIZON: int = 10

# RTC 추론 동안 이전 chunk에서 실행되는 step 수, RTC guidance 추론이 약 130ms(30Hz 약 4 step)라 여유를 둔다.
RTC_INFERENCE_DELAY: int = 5

TASK_PROMPT: str = 'pick up the white Ethernet cable and plug it into the black Ethernet port'

EPISODE_STEPS: int = 30 * 120

SAMPLE_FRAME_COUNT: int = 4

MAX_JOINT_STEP_RAD: float = 0.1

POLICY_SERVER_HOST: str = '127.0.0.1'
POLICY_SERVER_PORT: int = 8000

POLICY_CONFIG_NAME: str = 'pi05_yam_dit_lora'
SERVER_START_TIMEOUT_S: float = 300.0
CONTROL_HZ: float = 30.0

FOLLOWER_CAN_CHANNEL: str = 'can0'
GRIPPER_TYPE: str = 'linear_4310'
EPISODE_INITIAL_POSE: tuple[float, ...] = (0.0, 0.8, 0.8, 0.0, 0.0, 0.0, 1.0)

TOP_CAMERA_DEVICE_PATH: str = '/dev/v4l/by-id/usb-Intel_R__RealSense_TM__Camera_Intel_R__RealSense_TM__515_00000000F0244932-video-index0'
TOP_CAMERA_CAPTURE_WIDTH: int = 960
TOP_CAMERA_CAPTURE_HEIGHT: int = 540
TOP_CAMERA_CAPTURE_FPS: int = 30
TOP_CAMERA_PIXEL_FORMAT: str = 'YUYV'
TOP_CAMERA_FIT_MODE: CameraFitMode = 'center_crop'

WRIST_CAMERA_DEVICE_PATH: str = '/dev/v4l/by-path/pci-0000:e3:00.4-usb-0:2:1.3-video-index0'
WRIST_CAMERA_CAPTURE_WIDTH: int = 640
WRIST_CAMERA_CAPTURE_HEIGHT: int = 480
WRIST_CAMERA_CAPTURE_FPS: int = 30
WRIST_CAMERA_PIXEL_FORMAT: str = 'YUYV'
WRIST_CAMERA_FIT_MODE: CameraFitMode = 'center_crop'

CAMERA_OUTPUT_SIZE: int = 224
CAMERA_SETTLE_TIME_S: float = 1.0
CAMERA_MAX_FRAME_AGE_S: float = 0.5

## 이미지와 session 구성

MuJoCo에서는 데이터셋 image provider, 실제 YAM에서는 위 camera 경로를 선택한다.

In [ ]:
# MuJoCo에서는 데이터셋의 동기화된 top/wrist RGB frame을 반복 공급한다.
sample_image_provider: ImageProvider | None = None
if EXECUTION_TARGET == 'mujoco':
    top_video_path: Path = DATASET_DIR / 'videos' / 'observation.images.top_rgb' / 'chunk-000' / 'file-000.mp4'
    wrist_video_path: Path = DATASET_DIR / 'videos' / 'observation.images.wrist_rgb' / 'chunk-000' / 'file-000.mp4'
    with av.open(str(top_video_path)) as top_video, av.open(str(wrist_video_path)) as wrist_video:
        top_frames: Iterator[av.VideoFrame] = top_video.decode(video=0)
        wrist_frames: Iterator[av.VideoFrame] = wrist_video.decode(video=0)
        frame_pairs: tuple[dict[str, np.ndarray], ...] = tuple(
            {
                'top': top_frame.to_ndarray(format='rgb24'),
                'wrist': wrist_frame.to_ndarray(format='rgb24'),
            }
            for top_frame, wrist_frame in islice(zip(top_frames, wrist_frames, strict=True), SAMPLE_FRAME_COUNT)
        )
    if len(frame_pairs) != SAMPLE_FRAME_COUNT:
        raise ValueError('dataset does not contain enough paired frames')
    frame_iterator: Iterator[dict[str, np.ndarray]] = cycle(frame_pairs)

    def next_sample_images(
    ) -> Mapping[str, object]:
        '''다음 top/wrist RGB frame pair를 MuJoCo observation에 넣어 반환한다.'''
        return next(frame_iterator)

    sample_image_provider = next_sample_images

# 실제 YAM은 기존 CameraRig를 사용하고 MuJoCo는 dataset image provider를 사용한다.
camera_devices: tuple[CameraDeviceConfig, ...] = ()
if EXECUTION_TARGET == 'real':
    camera_devices = (
        CameraDeviceConfig(
            role='top',
            device_path=TOP_CAMERA_DEVICE_PATH,
            capture_width=TOP_CAMERA_CAPTURE_WIDTH,
            capture_height=TOP_CAMERA_CAPTURE_HEIGHT,
            capture_fps=TOP_CAMERA_CAPTURE_FPS,
            pixel_format=TOP_CAMERA_PIXEL_FORMAT,
            fit_mode=TOP_CAMERA_FIT_MODE,
            output_size=CAMERA_OUTPUT_SIZE,
        ),
        CameraDeviceConfig(
            role='wrist',
            device_path=WRIST_CAMERA_DEVICE_PATH,
            capture_width=WRIST_CAMERA_CAPTURE_WIDTH,
            capture_height=WRIST_CAMERA_CAPTURE_HEIGHT,
            capture_fps=WRIST_CAMERA_CAPTURE_FPS,
            pixel_format=WRIST_CAMERA_PIXEL_FORMAT,
            fit_mode=WRIST_CAMERA_FIT_MODE,
            output_size=CAMERA_OUTPUT_SIZE,
        ),
    )
camera_config: CameraConfig = CameraConfig(
    devices=camera_devices,
    settle_time_s=CAMERA_SETTLE_TIME_S,
    max_frame_age_s=CAMERA_MAX_FRAME_AGE_S,
)

# 실제 YAM은 연결 확인용으로 camera를 열어 frame 하나를 읽고 해제한다.
if EXECUTION_TARGET == 'real':
    camera_rig: CameraRig = CameraRig(camera_config)
    camera_rig.connect()
    try:
        camera_frames: dict[str, CameraFrame] = camera_rig.read_frames()
    finally:
        camera_rig.close()
    camera_role: str
    camera_frame: CameraFrame
    for camera_role, camera_frame in camera_frames.items():
        display({'role': camera_role, 'shape': camera_frame.images['rgb'].shape})
        display(Image.fromarray(camera_frame.images['rgb']))
robot_config: RobotConfig = RobotConfig(
    follower_can_channel=FOLLOWER_CAN_CHANNEL,
    gripper_type=GRIPPER_TYPE,
    episode_initial_pose=EPISODE_INITIAL_POSE,
)
config: RunConfig = build_run_config(
    mode='inference',
    teleop_source='quest3',
    vla_type='pi0.5',
    checkpoint_uri=str(CHECKPOINT_DIR),
    checkpoint_revision=None,
    policy_config_name=POLICY_CONFIG_NAME,
    use_rtc=USE_RTC,
    execution_target=EXECUTION_TARGET,
    use_safety_gate=EXECUTION_TARGET == 'real',
    control_hz=CONTROL_HZ,
    task_prompt=TASK_PROMPT,
    data_root='data/episodes',
    robot=robot_config,
    quest=QuestConfig(display_mode='none'),
    camera=camera_config,
)

class ActionClipProducer:
    '''OpenPI action을 teleop 데이터 수집과 같은 범위로 clip한 뒤 safety gate에 전달한다.'''

    def __init__(
        self,
        producer: ActionProducer,
    ) -> None:
        '''감쌀 OpenPI action producer를 저장한다.'''
        self._producer: ActionProducer = producer

    def connect(
        self,
    ) -> None:
        '''OpenPI action producer를 연결한다.'''
        self._producer.connect()

    def reset(
        self,
        observation: RobotObservation,
    ) -> None:
        '''OpenPI server에 다시 연결한 뒤 새 episode의 measured state로 reset한다.'''
        # Server가 종료된 상태에서 재연결을 무한히 기다리지 않도록 먼저 확인한다.
        if server_process.poll() is not None:
            raise RuntimeError('OpenPI server has exited; rerun the server cell')
        # 셀 정지로 응답을 받기 전에 중단된 이전 inference가 다음 episode에 섞이지 않도록 연결을 새로 연다.
        self._producer.close()
        self._producer.connect()
        self._producer.reset(observation)

    def next_action(
        self,
        observation: RobotObservation,
    ) -> RobotAction:
        '''OpenPI action의 joint 변화량과 gripper 값을 clip하여 반환한다.'''
        action: RobotAction = self._producer.next_action(observation)
        # 학습 데이터처럼 joint target을 measured state 기준 ±MAX_JOINT_STEP_RAD 안으로 제한한다.
        # 부동소수점 오차로 safety gate의 경계값을 넘지 않도록 1e-9 rad 안쪽으로 clip한다.
        max_step_rad: float = MAX_JOINT_STEP_RAD - 1e-9
        joints: tuple[float, ...] = tuple(
            min(max(target, current - max_step_rad), current + max_step_rad)
            for current, target in zip(observation.state[:6], action.values[:6], strict=True)
        )
        # 정규화 복원 오차로 1.0을 조금 넘는 gripper 값이 safety gate에서 거부되지 않도록 한다.
        gripper: float = min(max(action.values[6], 0.0), 1.0)
        return RobotAction(values=(*joints, gripper))

    def close(
        self,
    ) -> None:
        '''OpenPI action producer를 닫는다.'''
        self._producer.close()

    def __getattr__(
        self,
        name: str,
    ) -> object:
        '''diagnostics 등 optional attribute는 OpenPI action producer에 위임한다.'''
        return getattr(self._producer, name)


class LoggingSafetyGate:
    '''Safety gate가 action을 거부하면 이유와 state, action을 출력한다.'''

    def __init__(
        self,
        gate: SafetyGate,
    ) -> None:
        '''감쌀 safety gate를 저장한다.'''
        self._gate: SafetyGate = gate

    def reset(
        self,
    ) -> None:
        '''감싼 safety gate를 reset한다.'''
        self._gate.reset()

    def evaluate(
        self,
        observation: RobotObservation,
        action: RobotAction,
    ) -> SafetyDecision:
        '''감싼 safety gate의 판정을 반환하고 거부 시 원인 값을 출력한다.'''
        decision: SafetyDecision = self._gate.evaluate(observation, action)
        if not decision.accepted:
            joint_steps: tuple[float, ...] = tuple(
                abs(target - current)
                for current, target in zip(observation.state[:6], action.values[:6], strict=True)
            )
            display({
                'reason': decision.reason,
                'state': observation.state,
                'action': action.values,
                'joint_steps': joint_steps,
                'max_joint_step_rad': MAX_JOINT_STEP_RAD,
            })
        return decision


def create_safety_gate(
    run_config: RunConfig,
) -> SafetyGate:
    '''RunConfig를 받아 실제 YAM의 policy joint step 제한을 생성한다.'''
    del run_config
    return LoggingSafetyGate(JointStepSafetyGate(max_joint_step_rad=MAX_JOINT_STEP_RAD))

robot_factory: RobotFactory = (
    partial(create_default_robot, image_provider=sample_image_provider)
    if EXECUTION_TARGET == 'mujoco'
    else create_default_robot
)
def create_clipped_pi_action_producer(
    run_config: RunConfig,
) -> ActionProducer:
    '''OpenPI action producer를 생성하고 action clip으로 감싼다.'''
    producer: ActionProducer = create_pi_action_producer(
        run_config,
        host=POLICY_SERVER_HOST,
        port=POLICY_SERVER_PORT,
        rtc_inference_delay=RTC_INFERENCE_DELAY,
        rtc_execute_horizon=RTC_EXECUTE_HORIZON,
        open_loop_execute_steps=OPEN_LOOP_EXECUTE_STEPS,
    )
    return ActionClipProducer(producer)

pi_factory: ActionProducerFactory = create_clipped_pi_action_producer
dependencies: RuntimeDependencies = RuntimeDependencies(
    robot_factory=robot_factory,
    pi_factory=pi_factory,
    rtc_factory=pi_factory,
    safety_gate_factory=create_safety_gate,
)
session: RunSession = create_session(config=config, dependencies=dependencies)

## 1. OpenPI server 시작

이 셀은 checkpoint를 로드하고 첫 inference로 JAX를 준비한다.

In [ ]:
# 현재 step checkpoint의 params와 norm_stats를 확인한다.
assert (CHECKPOINT_DIR / 'params' / '_METADATA').is_file()
assert (CHECKPOINT_DIR / 'assets' / 'yam' / 'norm_stats.json').is_file()
if 'server_process' in globals() and server_process.poll() is None:
    raise RuntimeError('OpenPI server is already running in this notebook')
with socket.socket() as probe_socket:
    probe_socket.settimeout(1.0)
    if probe_socket.connect_ex((POLICY_SERVER_HOST, POLICY_SERVER_PORT)) == 0:
        raise RuntimeError('policy server port is already in use')

# OpenPI server는 episode와 별도 process로 계속 실행한다.
server_command: list[str] = [
    str(OPENPI_DIR / '.venv' / 'bin' / 'python'),
    'scripts/serve_policy.py',
    '--port',
    str(POLICY_SERVER_PORT),
    'policy:checkpoint',
    '--policy.config',
    POLICY_CONFIG_NAME,
    '--policy.dir',
    str(CHECKPOINT_DIR),
]
# VS Code의 셀 정지 버튼은 kernel의 모든 자식 process에 SIGINT를 보내므로 server는 SIGINT를 무시한 상태로 실행한다.
# 무시 설정은 자식 process에 상속되고, server는 마지막 셀의 terminate(SIGTERM)로만 종료된다.
previous_sigint_handler = signal.signal(signal.SIGINT, signal.SIG_IGN)
try:
    server_process: subprocess.Popen[bytes] = subprocess.Popen(server_command, cwd=OPENPI_DIR, start_new_session=True)
finally:
    signal.signal(signal.SIGINT, previous_sigint_handler)
deadline_s: float = time.monotonic() + SERVER_START_TIMEOUT_S
while time.monotonic() < deadline_s:
    if server_process.poll() is not None:
        raise RuntimeError('OpenPI server exited before accepting connections')
    server_ready: bool = False
    with socket.socket() as probe_socket:
        probe_socket.settimeout(1.0)
        if probe_socket.connect_ex((POLICY_SERVER_HOST, POLICY_SERVER_PORT)) == 0:
            probe_socket.sendall(b'GET /healthz HTTP/1.1\r\nHost: localhost\r\nConnection: close\r\n\r\n')
            server_ready = b'200 OK' in probe_socket.recv(128)
    if server_ready:
        break
    time.sleep(1.0)
else:
    raise TimeoutError('OpenPI server did not become ready')

# Robot 연결 전에 dummy observation으로 첫 JAX inference를 완료하고 action은 폐기한다.
warmup_image: np.ndarray = np.zeros((224, 224, 3), dtype=np.uint8)
warmup_observation: RobotObservation = RobotObservation(
    state=EPISODE_INITIAL_POSE,
    images={'top': warmup_image, 'wrist': warmup_image},
)
warmup_backend: WebsocketPiBackend = WebsocketPiBackend(
    host=POLICY_SERVER_HOST,
    port=POLICY_SERVER_PORT,
    prompt=TASK_PROMPT,
)
warmup_backend.connect()
try:
    warmup_chunk: ActionChunk = warmup_backend.predict_chunk(warmup_observation)
    assert warmup_chunk.horizon > 0
    if USE_RTC:
        assert warmup_chunk.horizon > RTC_EXECUTE_HORIZON
        # RTC guidance용 JAX compile(약 10초)도 실제 episode 전에 수행한다.
        warmup_rtc: RtcGuidance = RtcGuidance(
            prev_action_chunk=ActionChunk(values=warmup_chunk.values[RTC_EXECUTE_HORIZON:]),
            inference_delay=RTC_INFERENCE_DELAY,
            prefix_attention_horizon=warmup_chunk.horizon - RTC_EXECUTE_HORIZON,
        )
        warmup_rtc_chunk: ActionChunk = warmup_backend.predict_chunk(warmup_observation, warmup_rtc)
        assert warmup_rtc_chunk.horizon == warmup_chunk.horizon
finally:
    warmup_backend.close()


## 2. Session 연결

In [ ]:
# 선택한 execution target과 OpenPI action source를 연결한다.
session.connect()

## 3. Episode 초기화

로봇을 `episode_initial_pose`로 준비한다. `real`에서는 로봇이 움직이므로 작업 영역을 확인한 뒤 실행한다. Episode 실행 셀을 반복할 때도 같은 초기화가 자동 수행된다.

In [ ]:
# 실제 YAM은 human reset pose로 이동하고 MuJoCo는 environment를 reset한다.
session.prepare_episode()

## 4. Episode 실행

이 셀은 episode 하나만 실행한다. 원하는 횟수만큼 다시 실행할 수 있으며, 서버와 session은 유지된다. 매번 환경을 초기화하고 `EPISODE_STEPS`만큼 실행한다.

In [ ]:
# 종료된 episode를 다시 실행할 때 초기 상태로 준비한다.
if session.state != EpisodeState.PREPARED:
    session.prepare_episode()
state_before: tuple[float, ...] = session.get_observation().state
episode_result: EpisodeState = session.run_prepared_episode(max_steps=EPISODE_STEPS)
state_after: tuple[float, ...] = session.get_observation().state
display(episode_result)
display({'state_before': state_before, 'state_after': state_after})
display(session.control_loop_diagnostics())

## 5. 그리퍼 열기

현재 관절 자세를 유지하면서 그리퍼만 연다.

In [ ]:
# 현재 측정 자세에서 gripper만 완전히 연 pose로 이동한다.
GRIPPER_OPEN: float = 1.0
current_state: tuple[float, ...] = session._robot.get_observation().state
open_gripper_pose: tuple[float, ...] = (*current_state[:6], GRIPPER_OPEN)
display({'current': current_state, 'target': open_gripper_pose})
session._robot.move_to_pose(open_gripper_pose)

## 6. Rest pose

종료 전 로봇을 rest pose로 이동한다.

In [ ]:
# 로봇을 안전한 rest pose로 이동한다.
REST_POSE: tuple[float, ...] = (0.0, 0.0, 0.02, 0.0, 0.0, 0.0, 1.0)
session._robot.move_to_pose(REST_POSE)

## 7. Session과 OpenPI server 종료

모든 episode가 끝난 뒤 실행한다. 다음 사용 시에는 server 시작부터 다시 실행한다.

In [ ]:
# Action source와 robot resource를 닫은 뒤 서버를 중지한다.
session.close()
if server_process.poll() is None:
    server_process.terminate()
server_process.wait(timeout=15)